# A100 での報酬信号と昇格ゲート

研究上の問い: 正しい報酬だけが held-out の精度を上げるのか。その差はモデル族によるのか。自己検証器を報酬にしたとき、報酬ハッキングを検出して候補を戻せるか。

Part A は整数式の GRPO で、報酬を correctness / random / incorrect / format / base の 5 本にする。random は難易度プローブの正解率を p とする Bernoulli で、生成文とは独立に引く。incorrect は厳密な最終行が誤りなら 1。format は `#### <integer>` が解析できれば 1。base は同じログを書くが optimizer は動かさない。KL 係数は 0。参照モデルは KL を記録するためだけに載せる。base の KL は 0 で、参照の forward はしない。

モデルは `Qwen/Qwen2.5-1.5B-Instruct` と `meta-llama/Llama-3.2-1B-Instruct`。Llama の重みが読めなければ `HuggingFaceTB/SmolLM2-1.7B-Instruct` に切り替え、理由を出す。更新は最終層の LoRA（rank 8、`q_proj` と `v_proj`、bf16）。生成は vLLM を使わず、Hugging Face `generate` を左パディングでバッチする。1 バッチの torch seed は 1 つ。

難易度は dev 16 問の greedy プローブで 2, 3, 4 から選ぶ。正解率が 0.2 から 0.5 ならその中で 0.35 に近いもの、無ければ最も近いものを選び `calibration_inside_band: false` と出す。confirm は読まない。held-out は GSM8K test の部分集合。主指標は厳密な最終行。最後の整数の一致は記録するだけで採否に使わない。

FULL は 1 台の A100 で 2.5 時間以内に収める。QUICK の目安は 15–20 分で、`throughput.json` に秒/サンプルを書く。未計測の prior（学習 0.35 秒、評価 0.20 秒）は測定ではない。FULL はそのファイルがあれば読み、ステップを 20 ずつ、下限 40 まで減らし、それでも超えるときは held-out を減らして理由を出す。`run.json` の後はステップ数を変えない。Drive の JSONL と checkpoint から再開する。切断で失うのは、書き終えていない 1 ステップか、書き終えていない 1 回の eval まで。

Part B は Qwen2.5-1.5B だけ、FULL で 3 ラウンド、QUICK で 2 ラウンド。報酬は式の再計算か、モデル自身の検証器。検証器は `#### <integer>` を書き、その整数が候補と一致すれば受理する。コードとしては実行しない。dev は 32 問なので `PromotionGate` には `min_eval_examples=32` を渡し、paired bootstrap はオフにする。confirm は足さない。CPU フィクスチャが、自己報酬が上がり dev 正解率が下がり false-accept が 0.20 を超えるパターンの拒否を示す。GPU の図は、その短い実行でゲートが何を返したかの記録である。held-out の能力向上は主張しない。

図の読み方: Part A は held-out の厳密正解率で、帯は seed の最小と最大。Part B はラウンドごとの自己報酬と dev 正解率で、点に accept または reject が付く。自己報酬が上がり正解率が横ばいか下がり、false-accept が上がり、reject なら、その実行ではゲートが候補を戻した。


In [ ]:
import os

# Pins observed on the hub on 2026-10-13. The runner records the one it loads.
# Qwen/Qwen2.5-1.5B-Instruct 989aa7980e4cf806f80c7fef2b1adb7bc71aa306
# meta-llama/Llama-3.2-1B-Instruct 9213176726f574b556790deb65791e0c5aa438b6
# HuggingFaceTB/SmolLM2-1.7B-Instruct 31b70e2e869a7173562077fd711b654946d38674
MODE = "full"  # "quick" or "full"
os.environ["MODE"] = MODE
os.environ["VARE_REPO"] = "https://github.com/mitukx/VARE.git"
os.environ["VARE_REV"] = "cursor/minigrpo-colab-af75"
print("mode", MODE)


In [ ]:
import os
import subprocess
from pathlib import Path

repo = Path("/content/vare-flagship")
if not (repo / ".git").exists():
    subprocess.check_call(["git", "clone", os.environ["VARE_REPO"], str(repo)])
subprocess.check_call(["git", "fetch", "origin", os.environ["VARE_REV"]], cwd=repo)
subprocess.check_call(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=repo)
print(subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip())


In [ ]:
import subprocess
import sys

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        "/content/vare-flagship",
        "transformers",
        "accelerate",
        "peft",
        "huggingface_hub",
        "datasets",
        "matplotlib",
    ]
)


In [ ]:
import os
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
root = Path("/content/drive/MyDrive/vare/flagship-a100")
output = root / MODE
output.mkdir(parents=True, exist_ok=True)
os.environ["FLAGSHIP_OUTPUT"] = str(output)
os.environ["FLAGSHIP_THROUGHPUT"] = str(root / "throughput.json")
os.environ["HF_HOME"] = str(root / "huggingface-cache")
os.environ["HF_HUB_CACHE"] = str(root / "huggingface-cache" / "hub")
# Llama-3.2 is gated. If this runtime has an accepted license, set HF_TOKEN before the run.
print("output", output)
print("throughput", os.environ["FLAGSHIP_THROUGHPUT"])


次のセルは GPU 名と見積もりだけを出す。FULL のステップ数は、この見積もりか、既にある `throughput.json` から決まる。prior の秒数は測定ではない。その次のセルが学習を始める。


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import torch

if not torch.cuda.is_available():
    raise SystemExit("CUDA is required. This notebook does not record a fake GPU result.")
name = torch.cuda.get_device_name(0)
print("gpu", name)
if "A100" not in name:
    print("warning: the 2.5h grid targets one A100; continuing on this GPU")
repo = Path("/content/vare-flagship")
planned = subprocess.run(
    [
        sys.executable,
        str(repo / "scripts/run_flagship.py"),
        "--mode",
        MODE,
        "--plan",
        "--throughput",
        os.environ["FLAGSHIP_THROUGHPUT"],
    ],
    cwd=repo,
    check=False,
    capture_output=True,
    text=True,
)
print(planned.stdout)
if planned.returncode != 0:
    print(planned.stderr)
    raise SystemExit(planned.returncode)
report = json.loads(planned.stdout)
print(
    "eta_hours",
    round(float(report["eta_seconds"]) / 3600, 3),
    "steps",
    report["steps"],
    "heldout",
    report["heldout_n"],
    "rate_source",
    report["rate_source"],
)


学習セル。同じ `output` でもう一度実行すると、完了したステップは飛ばす。QUICK を先に終えると、別ディレクトリの FULL がその throughput を読む。


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

repo = Path("/content/vare-flagship")
completed = subprocess.run(
    [
        sys.executable,
        str(repo / "scripts/run_flagship.py"),
        "--mode",
        MODE,
        "--output",
        os.environ["FLAGSHIP_OUTPUT"],
        "--throughput",
        os.environ["FLAGSHIP_THROUGHPUT"],
    ],
    cwd=repo,
)
if completed.returncode != 0:
    raise SystemExit(completed.returncode)


In [ ]:
import json
import os
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt

output = Path(os.environ["FLAGSHIP_OUTPUT"])
rows = []
for line in (output / "metrics.jsonl").read_text(encoding="utf-8").splitlines():
    if line.strip():
        rows.append(json.loads(line))
heldout = [row for row in rows if row.get("kind") == "eval" and row.get("split") == "heldout"]
by_model = defaultdict(list)
for row in heldout:
    by_model[row["model"]].append(row)
colors = {
    "correctness": "#1f4e79",
    "random": "#b36b00",
    "incorrect": "#9b2226",
    "format": "#2d6a4f",
    "base": "#6c757d",
}
figure, axes = plt.subplots(len(by_model) or 1, 1, figsize=(10, 3.2 * max(1, len(by_model))), squeeze=False)
for axis, (model, model_rows) in zip(axes.ravel(), by_model.items() or [("none", [])]):
    for arm, color in colors.items():
        steps = sorted({int(row["step"]) for row in model_rows if row["arm"] == arm})
        lows, highs, means = [], [], []
        for step in steps:
            values = [float(row["accuracy"]) for row in model_rows if row["arm"] == arm and int(row["step"]) == step]
            if not values:
                continue
            lows.append(min(values))
            highs.append(max(values))
            means.append(sum(values) / len(values))
        if not means:
            continue
        axis.fill_between(steps[: len(means)], lows, highs, color=color, alpha=0.15)
        axis.plot(steps[: len(means)], means, color=color, label=arm)
    axis.set_ylim(0, 1)
    axis.set_title(model)
    axis.set_xlabel("step")
    axis.set_ylabel("held-out accuracy")
    axis.legend(loc="best", fontsize=8)
figure.tight_layout()
figure_path = output / "figures" / "heldout.png"
figure_path.parent.mkdir(parents=True, exist_ok=True)
figure.savefig(figure_path)
print("saved", figure_path)
rsi = [row for row in rows if row.get("kind") in {"rsi_baseline", "rsi_round"}]
print("reward_source round self_reward true_accuracy accepted false_accept_rate")
for row in sorted(rsi, key=lambda item: (str(item["reward_source"]), int(item["round"]))):
    print(
        row["reward_source"],
        row["round"],
        row["self_reward"],
        row["true_accuracy"],
        row.get("accepted"),
        row.get("false_accept_rate"),
    )
